# Test Google Drive data access

Run this first with the hosted Colab kernel. It mounts Drive, locates both datasets, counts representative files, and opens one image from each dataset. It does not modify data.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

print("Python:", sys.version)
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed.")


In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT))

# Change only this value if the Drive project is moved.
PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "ITU" / "3D" / "Thesis"

def find_unique_dir(names, search_roots):
    direct = [root / name for root in search_roots for name in names]
    matches = [p for p in direct if p.is_dir()]
    if not matches:
        for root in search_roots:
            if root.is_dir():
                matches.extend(p for p in root.rglob("*") if p.is_dir() and p.name.casefold() in {n.casefold() for n in names})
    unique = list(dict.fromkeys(p.resolve() for p in matches))
    if len(unique) != 1:
        raise FileNotFoundError(f"Expected one of {names}; found {len(unique)}: {unique}")
    return unique[0]

SEARCH_ROOTS = [PROJECT_ROOT / "data", PROJECT_ROOT]
INDUSTRIAL_ROOT = find_unique_dir(["IndustrialInventory"], SEARCH_ROOTS)
HQ200_ROOT = find_unique_dir(["3DrealCarHQ200", "HQ200"], SEARCH_ROOTS)

# If HQ200 is a wrapper folder, descend to the folder containing capture scenes.
if (HQ200_ROOT / "3DrealCarHQ200").is_dir():
    HQ200_ROOT = HQ200_ROOT / "3DrealCarHQ200"

print("Project:   ", PROJECT_ROOT)
print("Industrial:", INDUSTRIAL_ROOT)
print("3DRealCar: ", HQ200_ROOT)


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

def image_files(root):
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

industrial_images = image_files(INDUSTRIAL_ROOT)
hq_images = sorted(HQ200_ROOT.rglob("frame_*.jpg"))
if not hq_images:
    hq_images = image_files(HQ200_ROOT)

summary = {
    "Industrial images": len(industrial_images),
    "3DRealCar RGB images": len(hq_images),
    "3DRealCar camera JSON": len(list(HQ200_ROOT.rglob("frame_*.json"))),
    "3DRealCar depth maps": len(list(HQ200_ROOT.rglob("depth_*.png"))),
    "3DRealCar meshes": len(list(HQ200_ROOT.rglob("*.obj"))),
}
for label, count in summary.items():
    print(f"{label:28s}: {count:,}")

if not industrial_images or not hq_images:
    raise FileNotFoundError("At least one dataset has no discoverable RGB images. Check the printed roots.")


In [ ]:
examples = [("IndustrialInventory", industrial_images[0]), ("3DRealCar", hq_images[0])]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (dataset, path) in zip(axes, examples):
    with Image.open(path) as image:
        rgb = image.convert("RGB")
        ax.imshow(rgb)
        ax.set_title(f"{dataset}\n{rgb.width} × {rgb.height}\n{path.name}")
    ax.axis("off")
    print(dataset, "read OK:", path)
plt.tight_layout()
plt.show()
print("Drive access test passed.")
